In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.config import *

In [0]:
for endpoint in VALID_ENDPOINTS:
    display(endpoint)

In [0]:
users = spark.read.table(f'{TABLE_PREFIX}.bronze_users')

In [0]:
users = users.select(
    F.col('id').alias('user_id'),
    F.col('firstName').alias('first_name'),
    F.col('lastName').alias('last_name'),
    F.col('maidenName').alias('maiden_name'),
    'age',
    'gender',
    'email',
    'phone',
    'username',
    'password',
    F.to_date(F.col('birthDate'), 'yyyy-M-d').alias('birth_date'),
    'image',
    F.col('bloodGroup').alias('blood_group'),
    'height',
    'weight',
    F.col('eyeColor').alias('eye_color'),
    F.col('hair.color').alias('hair_color'),
    F.col('hair.type').alias('hair_type'),
    'ip',
    F.col('address.address').alias('street_address'),
    F.col('address.city').alias('city'),
    F.col('address.state').alias('state'),
    F.col('address.stateCode').alias('state_code'),
    F.col('address.postalCode').alias('postal_code'),
    F.col('address.coordinates.lat').alias('lat_code'),
    F.col('address.coordinates.lng').alias('long_code'),
    F.col('address.country').alias('country'),
    F.col('macAddress').alias('mac_address'),
    'university',
    F.last_day(F.to_date(F.col('bank.cardExpire'), 'MM/yy')).alias('card_expire'),
    F.col('bank.cardNumber').alias('card_number'),
    F.col('bank.cardType').alias('card_type'),
    F.col('bank.currency').alias('currency'),
    F.col('bank.iban').alias('iban'),
    F.col('company.department').alias('company_dept'),
    F.col('company.name').alias('company_name'),
    F.col('company.title').alias('job_title'),
    F.col('company.address.address').alias('office_street_address'),
    F.col('company.address.city').alias('office_city'),
    F.col('company.address.state').alias('office_state'),
    F.col('company.address.stateCode').alias('office_state_code'),
    F.col('company.address.postalCode').alias('office_postal_code'),
    F.col('company.address.coordinates.lat').alias('office_lat'),
    F.col('company.address.coordinates.lng').alias('office_long'),
    F.col('company.address.country').alias('office_country'),
    'ein',
    'ssn',
    F.col('userAgent').alias('user_agent'),
    F.col('crypto.coin').alias('crypto_coin'),
    F.col('crypto.wallet').alias('crypto_wallet'),
    F.col('crypto.network').alias('crypto_network'),
    'role',
    'ingestion_timestamp',
    'source_system'
)

In [0]:
cart_items = spark.read.table(f'{TABLE_PREFIX}.bronze_carts')

In [0]:
cart_items = cart_items.withColumn('products', F.explode(F.col('products')))

In [0]:
cart_items = cart_items.select(
    F.col('id').alias('cart_id'),
    F.col('products.id').alias('product_id'),
    F.col('products.title').alias('title'),
    F.col('products.price').alias('unit_price'),
    F.col('products.quantity').alias('quantity'),
    F.col('products.total').alias('line_total'),
    F.col('products.discountPercentage').alias('line_discount_percentage'),
    F.col('products.discountedTotal').alias('line_discounted_total'),
    F.col('products.thumbnail').alias('thumbnail'),
    F.col('total').alias('cart_total'),
    F.col('discountedTotal').alias('cart_discounted_total'),
    F.col('userId').alias('user_id'),
    F.col('totalProducts').alias('cart_total_products'),
    F.col('totalQuantity').alias('cart_total_quantity'),
    'ingestion_timestamp',
    'source_system'
)

In [0]:
products = spark.read.table(f'{TABLE_PREFIX}.bronze_products')

In [0]:
product_reviews = products.select('id', F.explode(F.col('reviews')).alias('reviews'))

In [0]:
products = products.select(
    F.col('id').alias('product_id'),
    'title',
    'description',
    'category',
    'price',
    F.col('discountPercentage').alias('discount_percentage'),
    'rating',
    'stock',
    'tags',
    'brand',
    'sku',
    'weight',
    F.col('dimensions.width').alias('width'),
    F.col('dimensions.height').alias('height'),
    F.col('dimensions.depth').alias('depth'),
    F.col('warrantyInformation').alias('warranty_information'),
    F.col('shippingInformation').alias('shipping_information'),
    F.col('availabilityStatus').alias('availability_status'),
    F.col('returnPolicy').alias('return_policy'),
    F.col('minimumOrderQuantity').alias('min_order_quantity'),
    F.col('meta.createdAt').alias('created_at'),
    F.col('meta.updatedAt').alias('updated_at'),
    F.col('meta.barCode').alias('bar_code'),
    F.col('meta.qrCode').alias('qr_code'),
    'thumbnail',
    'ingestion_timestamp',
    'source_system'
)

In [0]:
product_reviews = product_reviews.select(
    F.col('id').alias('product_id'),
    F.col('reviews.rating').alias('review_rating'),
    F.col('reviews.comment').alias('review_comment'),
    F.to_date(F.col('reviews.date')).alias('review_date'),
    F.col('reviews.reviewerName').alias('reviewer_name'),
    F.col('reviews.reviewerEmail').alias('reviewer_email'),
)

In [0]:
users.writeTo(f'{TABLE_PREFIX}.silver_users').createOrReplace()
products.writeTo(f'{TABLE_PREFIX}.silver_products').createOrReplace()
product_reviews.writeTo(f'{TABLE_PREFIX}.silver_product_reviews').createOrReplace()
cart_items.writeTo(f'{TABLE_PREFIX}.silver_cart_items').createOrReplace()